# Relevance Judgment with **Early Interaction** (Decoder-only LM) on SQuAD Dataset

  
We'll treat relevance as a small **generative classification** task via log-probabilities:  
> *Given* `[instruction, question, passage]` → *predict* `Relevant` vs `Irrelevant` using next-token log-probs.

### Objectives
- What “early interaction” means (joint attention over *q* and *d* in a single sequence).
- How to create positive/negative examples from **SQuAD v1.1**.




## Early vs. Late Interaction (Quick refresher)
- **Late interaction (bi-encoder)**: encode question and passage *separately*, compare vectors (fast, scalable).
- **Early interaction (cross)**: let the model *see* question + passage together and decide relevance (richer token-level interactions, more accurate, slower).  
This notebook focuses on **early interaction using a decoder-only LM**: we create a prompt with instruction + question + passage and score the model’s preference for `Relevant` vs `Irrelevant`.


In [1]:
import torch
from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import torch.nn.functional as F
import random

# Helper to set a random seed for reproducibility
def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

set_seed(42)

###  Create positive & negative (question, passage) pairs

**Positives**: the original question-context pair (context contains the answer).  
**Negatives**: random contexts from other articles




In [2]:

# Load a small split of SQuAD dataset
squad = load_dataset("squad", split='validation[:500]')

# Creating the 'Positive' dataset
# These are the original, correct (question, context) pairs.
# We just add a 'label' column with the value 1.
positive_ds = squad.map(lambda example: {'label': 1})

# Creating the 'Negative' dataset ---

# Get a list of all contexts from the dataset
all_contexts = squad['context']

# Create a new list of contexts, shuffled, to serve as our negative (irrelevant) contexts.
# We make a copy and shuffle it.
shuffled_contexts = all_contexts[:] # Create a copy
random.shuffle(shuffled_contexts)

# Create the negative dataset by replacing the original context with a shuffled one.
# We take the original dataset and discard its 'context' column, then add the shuffled contexts.
negative_ds = squad.remove_columns('context').add_column('context', shuffled_contexts)

# Add a 'label' column with the value 0.
negative_ds = negative_ds.map(lambda example: {'label': 0})

# Important: It's possible a question was randomly paired with its original context.
# We filter these rare cases out to ensure all our negative examples are truly negative.
original_contexts = squad['context']
negative_ds = negative_ds.filter(
    lambda example, idx: example['context'] != original_contexts[idx],
    with_indices=True
)


#Combine Positive and Negative datasets

relevance_dataset = concatenate_datasets([positive_ds, negative_ds])

val_dataset = relevance_dataset.shuffle(seed=42)


print("Dataset creation complete.")
print(f"Total examples: {len(val_dataset)}")
print(f"Number of positive examples: {len(val_dataset.filter(lambda x: x['label'] == 1))}")
print(f"Number of negative examples: {len(val_dataset.filter(lambda x: x['label'] == 0))}")
print("\nExample from the final dataset:")
print(val_dataset[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:86: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset creation complete.
Total examples: 978


Filter:   0%|          | 0/978 [00:00<?, ? examples/s]

Number of positive examples: 500


Filter:   0%|          | 0/978 [00:00<?, ? examples/s]

Number of negative examples: 478

Example from the final dataset:
{'id': '56d2045de7d4791d009025f3', 'title': 'Super_Bowl_50', 'context': 'CBS broadcast Super Bowl 50 in the U.S., and charged an average of $5 million for a 30-second commercial during the game. The Super Bowl 50 halftime show was headlined by the British rock group Coldplay with special guest performers Beyoncé and Bruno Mars, who headlined the Super Bowl XLVII and Super Bowl XLVIII halftime shows, respectively. It was the third-most watched U.S. broadcast ever.', 'question': 'Who is the quarterback for the Panthers?', 'answers': {'text': ['Cam Newton', 'Cam Newton', 'Cam Newton'], 'answer_start': [77, 77, 77]}, 'label': 0}


#### Displaying the first positive and negative sample

In [13]:
# Find the first positive (label=1) and negative (label=0) example in our validation set
positive_example = val_dataset.filter(lambda example: example['label'] == 1)[5]
negative_example = val_dataset.filter(lambda example: example['label'] == 0)[0]

print("--- POSITIVE EXAMPLE (Ground Truth: Relevant) ---")
print(f"QUESTION: {positive_example['question']}")
print(f"CONTEXT: {positive_example['context'][:350]}...") # Show a snippet
print(f"LABEL: {positive_example['label']}")
print("\n" + "="*50 + "\n")
print("--- NEGATIVE EXAMPLE (Ground Truth: Irrelevant) ---")
print(f"QUESTION: {negative_example['question']}")
print(f"CONTEXT: {negative_example['context'][:350]}...") # Show a snippet
print(f"LABEL: {negative_example['label']}")

--- POSITIVE EXAMPLE (Ground Truth: Relevant) ---
QUESTION: how many yards did Newton get for passes in the 2015 season?
CONTEXT: The Panthers offense, which led the NFL in scoring (500 points), was loaded with talent, boasting six Pro Bowl selections. Pro Bowl quarterback Cam Newton had one of his best seasons, throwing for 3,837 yards and rushing for 636, while recording a career-high and league-leading 45 total touchdowns (35 passing, 10 rushing), a career-low 10 intercept...
LABEL: 1


--- NEGATIVE EXAMPLE (Ground Truth: Irrelevant) ---
QUESTION: Who is the quarterback for the Panthers?
CONTEXT: CBS broadcast Super Bowl 50 in the U.S., and charged an average of $5 million for a 30-second commercial during the game. The Super Bowl 50 halftime show was headlined by the British rock group Coldplay with special guest performers Beyoncé and Bruno Mars, who headlined the Super Bowl XLVII and Super Bowl XLVIII halftime shows, respectively. It was...
LABEL: 0


## 4. Load a decoder-only LM & tokenizer

Other models like cross-encoder-ms-marco-MiniLM-L-6-v2 can also be tried


In [14]:
# We'll use a distilled BERT model fine-tuned for sequence classification. Other models can be used from huggingface
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Set the model to evaluation mode
model.eval()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


##  Prompting format

We'll use a simple, consistent instruction and ask the model to answer




Inference for positive Example:

In [15]:
# Our "early interaction" format: [instruction, question, context]
instruction = "Is the following passage relevant to the question? "
text_input_positive = instruction + positive_example['question']

# Tokenize the input text and the context as a pair
inputs = tokenizer(text_input_positive, positive_example['context'], return_tensors="pt", truncation=True, padding=True)

# Perform inference
with torch.no_grad(): # Disable gradient calculation for efficiency
    logits = model(**inputs).logits

# The model outputs raw scores (logits). We convert them to probabilities using softmax.
probabilities = F.softmax(logits, dim=1).squeeze()
predicted_label = torch.argmax(probabilities).item()

print(f"--- PREDICTION ON POSITIVE EXAMPLE ---")
print(f"Question: {positive_example['question']}")
print(f"Context: {positive_example['context']}")
print(f"Model Output Logits: {logits.numpy().flatten()}")
print(f"Probabilities (Irrelevant, Relevant): {probabilities.numpy().round(4)}")
print(f"Predicted Label: {predicted_label} ({'Irrelevant' if predicted_label == 0 else 'Relevant'})")
print(f"Actual Label: {positive_example['label']} ({'Irrelevant' if positive_example['label'] == 0 else 'Relevant'})")

--- PREDICTION ON POSITIVE EXAMPLE ---
Question: how many yards did Newton get for passes in the 2015 season?
Context: The Panthers offense, which led the NFL in scoring (500 points), was loaded with talent, boasting six Pro Bowl selections. Pro Bowl quarterback Cam Newton had one of his best seasons, throwing for 3,837 yards and rushing for 636, while recording a career-high and league-leading 45 total touchdowns (35 passing, 10 rushing), a career-low 10 interceptions, and a career-best quarterback rating of 99.4. Newton's leading receivers were tight end Greg Olsen, who caught a career-high 77 passes for 1,104 yards and seven touchdowns, and wide receiver Ted Ginn, Jr., who caught 44 passes for 739 yards and 10 touchdowns; Ginn also rushed for 60 yards and returned 27 punts for 277 yards. Other key receivers included veteran Jerricho Cotchery (39 receptions for 485 yards), rookie Devin Funchess (31 receptions for 473 yards and five touchdowns), and second-year receiver Corey Brown (3

Inference for Negative Example

In [16]:
# Format the input text for the negative example
text_input_negative = instruction + negative_example['question']

# Tokenize
inputs = tokenizer(text_input_negative, negative_example['context'], return_tensors="pt", truncation=True, padding=True)

# Perform inference
with torch.no_grad():
    logits = model(**inputs).logits

# Convert to probabilities
probabilities = F.softmax(logits, dim=1).squeeze()
predicted_label = torch.argmax(probabilities).item()

print(f"--- PREDICTION ON NEGATIVE EXAMPLE ---")
print(f"Question: {negative_example['question']}")
print(f"Context: {negative_example['context']}")
print(f"Model Output Logits: {logits.numpy().flatten()}")
print(f"Probabilities (Irrelevant, Relevant): {probabilities.numpy().round(4)}")
print(f"Predicted Label: {predicted_label} ({'Irrelevant' if predicted_label == 0 else 'Relevant'})")
print(f"Actual Label: {negative_example['label']} ({'Irrelevant' if negative_example['label'] == 0 else 'Relevant'})")

--- PREDICTION ON NEGATIVE EXAMPLE ---
Question: Who is the quarterback for the Panthers?
Context: CBS broadcast Super Bowl 50 in the U.S., and charged an average of $5 million for a 30-second commercial during the game. The Super Bowl 50 halftime show was headlined by the British rock group Coldplay with special guest performers Beyoncé and Bruno Mars, who headlined the Super Bowl XLVII and Super Bowl XLVIII halftime shows, respectively. It was the third-most watched U.S. broadcast ever.
Model Output Logits: [ 0.87889737 -0.649363  ]
Probabilities (Irrelevant, Relevant): [0.8218 0.1782]
Predicted Label: 0 (Irrelevant)
Actual Label: 0 (Irrelevant)


Custom Example Inference

In [17]:
# Custom Example
example = {
    'question': "What is the capital of France?",
    'context': "The capital of France is Paris.",
    'label': 1  # relevant
}
text_input = instruction + example['question']


inputs = tokenizer(text_input, example['context'], return_tensors="pt", truncation=True, padding=True)

with torch.no_grad():
    logits = model(**inputs).logits


probabilities = F.softmax(logits, dim=1).squeeze()
predicted_label = torch.argmax(probabilities).item()

print(f"--- PREDICTION ON CUSTOM EXAMPLE ---")
print(f"Question: {example['question']}")
print(f"Context: {example['context']}")
print(f"Model Output Logits: {logits.numpy().flatten()}")
print(f"Probabilities (Irrelevant, Relevant): {probabilities.numpy().round(4)}")
print(f"Predicted Label: {predicted_label} ({'Irrelevant' if predicted_label == 0 else 'Relevant'})")
print(f"Actual Label: {example['label']} ({'Irrelevant' if example['label'] == 0 else 'Relevant'})")

--- PREDICTION ON CUSTOM EXAMPLE ---
Question: What is the capital of France?
Context: The capital of France is Paris.
Model Output Logits: [-0.04624769  0.21079731]
Probabilities (Irrelevant, Relevant): [0.4361 0.5639]
Predicted Label: 1 (Relevant)
Actual Label: 1 (Relevant)


Performance on Complete Dataset

In [20]:
def tokenize_function(examples):
    instruction = "Is the following passage relevant to the question? "

    questions_with_instruction = [instruction + q for q in examples['question']]
    return tokenizer(
        questions_with_instruction,
        examples['context'],
        truncation=True,
        padding='max_length',
        max_length=512
    )

print("Tokenizing the validation dataset...")
tokenized_val_dataset = val_dataset.map(tokenize_function, batched=True)
tokenized_val_dataset = tokenized_val_dataset.with_format("torch")
print("Tokenization complete.")

Tokenizing the validation dataset...


Map:   0%|          | 0/978 [00:00<?, ? examples/s]

Tokenization complete.


Computing the metrics

In [21]:
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)

    return {
        'accuracy': accuracy_score(p.label_ids, preds),
        'precision': precision_score(p.label_ids, preds),
        'recall': recall_score(p.label_ids, preds),
        'f1': f1_score(p.label_ids, preds)
    }

# We need TrainingArguments even if we are not training
training_args = TrainingArguments(
    output_dir='./results_eval_only',
    per_device_eval_batch_size=16,
    report_to="none"
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=tokenized_val_dataset,
    compute_metrics=compute_metrics,
)

# Call the evaluate method to get the final metrics
print("\n" + "="*50 + "\n")
print("Running evaluation on the entire validation set...")
evaluation_results = trainer.evaluate()

print("\n--- OVERALL EVALUATION RESULTS ---")
print(f"Accuracy:  {evaluation_results['eval_accuracy']:.4f}")
print(f"Precision: {evaluation_results['eval_precision']:.4f}")
print(f"Recall:    {evaluation_results['eval_recall']:.4f}")
print(f"F1-Score:  {evaluation_results['eval_f1']:.4f}")



Running evaluation on the entire validation set...



--- OVERALL EVALUATION RESULTS ---
Accuracy:  0.5092
Precision: 0.5266
Recall:    0.3960
F1-Score:  0.4521


### Advantages (early interaction / cross-encoders)

- Highest accuracy from full query–doc token interactions

- Disambiguates terms using document context

- Soft matches (synonyms/paraphrases) + positional cues




### Disadvantages

- Not scalable

- Latency/cost grow with candidate set size

- Limited input length; truncation can miss evidence

- Depends on upstream recall (can’t find unseen docs)
